In [ ]:
!pip install ucimlrepo

Problem Statement

The objective of this project is to classify steel plate defects into seven fault categories using machine learning techniques. This is a multiclass classification problem where each steel plate belongs to one of the following classes: Pastry, Z_Scratch, K_Scratch, Stains, Dirtiness, Bumps, and Other_Faults.

Fault Classes

- Pastry – Surface defect resembling pastry-like marks.
- Z_Scratch – Scratch shaped like the letter Z.
- K_Scratch – Scratch with K-like pattern.
- Stains – Surface discoloration or stains.
- Dirtiness – Dirt contamination on the steel surface.
- Bumps – Raised surface irregularities.
- Other_Faults – Miscellaneous fault categories.

The dataset contains 27 input features and 7 target classes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
steel_plates_faults = fetch_ucirepo(id=198)

# data (as pandas dataframes)
x = steel_plates_faults.data.features
y = steel_plates_faults.data.targets

# metadata
print(steel_plates_faults.metadata)

# variable information
print(steel_plates_faults.variables)

In [ ]:
print(x.shape)
print(y.shape)

In [ ]:
x.head()


In [ ]:
y.head()

In [ ]:
y = y.idxmax(axis=1)

Target Conversion using idxmax()

The original dataset contains seven binary target columns corresponding to the seven fault classes. Each sample belongs to only one fault category and therefore only one target column contains the value 1. The idxmax() function was used to convert these binary columns into a single multiclass target variable. This conversion is valid because each steel plate sample belongs to exactly one fault class.

In [ ]:
y.head()

In [ ]:
print(y.unique())

In [ ]:
print(y.value_counts())

In [ ]:
plt.figure(figsize=(8,5))
y.value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.ylabel("Count")
plt.show()

Class Distribution Analysis

The class distribution graph shows that the dataset is imbalanced. The Other_Faults class contains the highest number of samples, whereas Dirtiness and Stains contain the fewest samples. This imbalance can affect classification performance because machine learning models tend to learn majority classes more effectively than minority classes. Therefore, metrics such as Precision, Recall, and F1-score are also considered in addition to Accuracy.

In [ ]:
plt.figure(figsize=(18,14))

sns.heatmap(
    x.corr(),
    annot=True,
    fmt=".1f",
    cmap="coolwarm",
    annot_kws={"size":6}
)

plt.title("Correlation Heatmap")
plt.show()

Heatmap Observation

The correlation heatmap shows relationships between different features. Some geometric and luminosity-related features exhibit positive correlation, indicating that they contain similar information. Most features show weak to moderate correlation, suggesting that the dataset contains diverse information useful for fault classification.

In [ ]:
x.hist(figsize=(25,20))

plt.show()

In [ ]:
plt.figure(figsize=(20,8))

sns.boxplot(data=x)

plt.xticks(rotation=90)

plt.show()

Boxplot Observation

The boxplot reveals the presence of outliers in several features. These outliers indicate variations in defect characteristics among steel plates. The presence of outliers is expected in industrial datasets and reflects real-world variability in manufacturing processes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

x_train = scaler.fit_transform(X_train)
x_test = scaler.transform(X_test)

Feature Scaling

Feature scaling was performed using StandardScaler. Scaling is important because different features have different numerical ranges. KNN is a distance-based algorithm and requires scaling to ensure that all features contribute equally to distance calculations.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)

In [ ]:

y_pred_knn = knn.predict(x_test)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

KNN Model Analysis

The K-Nearest Neighbors (KNN) model achieved an accuracy of approximately 71.98%. Since KNN relies on distance calculations, feature scaling significantly improved its performance. However, the dataset contains overlapping fault classes and class imbalance, which limits KNN performance compared to ensemble methods.

In [ ]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)


In [ ]:

y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

Decision Tree Analysis

The Decision Tree model achieved an accuracy of approximately 71.72%. Decision Trees can model nonlinear relationships between features and do not require feature scaling. However, a single decision tree may overfit the training data, reducing generalization performance.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Analysis

The Random Forest model achieved the highest accuracy of approximately 77.38%. Random Forest is an ensemble learning algorithm that combines multiple decision trees and uses majority voting to determine the final prediction. This approach reduces overfitting and improves classification performance.

In [ ]:
print("KNN Classification Report")
print(classification_report(y_test, y_pred_knn))

print("Decision Tree Classification Report")
print(classification_report(y_test, y_pred_dt))

print("Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))

Classification Metrics

Accuracy measures the percentage of correctly classified samples.

Precision measures how many predicted samples of a class are actually correct.

Recall measures how many actual samples of a class are correctly identified.

F1-score is the harmonic mean of Precision and Recall.

Support represents the number of actual samples belonging to each class.

Macro Average calculates the metric equally across all classes.

Weighted Average calculates the metric while considering class frequencies.

Classification Report Analysis

The Random Forest model achieved the best overall performance. Classes such as K_Scratch, Z_Scratch, and Stains achieved high Precision, Recall, and F1-scores, indicating strong classification performance. Pastry showed relatively lower performance, suggesting overlap with other fault categories.

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(cm)

disp.plot()

plt.show()

Confusion Matrix Analysis

The confusion matrix provides a detailed view of model predictions. Diagonal values represent correctly classified samples, while off-diagonal values represent misclassifications.

For example, values such as 110, 77, and 37 indicate correctly classified samples for their respective fault categories. The concentration of values along the diagonal indicates that the Random Forest model performs well on most classes. Off-diagonal values indicate confusion between similar fault categories.

In [ ]:
results = {
    'KNN': accuracy_score(y_test, y_pred_knn),
    'Decision Tree': accuracy_score(y_test, y_pred_dt),
    'Random Forest': accuracy_score(y_test, y_pred_rf)
}

print(results)

In [ ]:
importance = rf.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': importance
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

In [ ]:
print(feature_importance.head(10))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance['Feature'][:10],
    feature_importance['Importance'][:10]
)

plt.title("Top 10 Important Features")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.show()

Feature Importance Analysis

Feature importance analysis was performed using Random Forest. The most influential features were Length_of_Conveyer, LogOfAreas, Log_X_Index, Sum_of_Luminosity, and Pixels_Areas. These features contributed significantly to distinguishing between different steel plate fault categories.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':[100,200,300],
    'max_depth':[None,10,20]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5
)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)
print("Best Score:",grid.best_score_)

Hyperparameter Tuning

Hyperparameter tuning was performed using GridSearchCV. Different combinations of Random Forest parameters were tested to identify the best-performing model. The optimal parameters were max_depth=None and n_estimators=200, achieving a cross-validation score of approximately 78.67%. This confirms that the selected Random Forest configuration is suitable for the dataset.

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(results.keys(), results.values())

plt.ylabel("Accuracy")

plt.title("Model Comparison")

plt.show()